In [2]:
import astropy.units as u
%load_ext autoreload
%autoreload 2
%aimport crossmatching_3d
from crossmatching_3d import Crossmatcher3D as Crossmatcher, allowed_3d_separation
from crossmatching import allowed_angular_separation
from astropy.io import ascii


In [4]:
# oddity_id = "TIC 419015728" #tau ceti
# oddity_id = "TIC 200385493" # Kapteyn
oddity_id = "TIC 349714736" # weird only 3d match

cm = Crossmatcher()
cm.load_catalog(from_file="pscomppars.txt")
input_t = ascii.read("./input/HPIC_LC4_combined_d50.txt")
cm.load_alternate_ids([oddity_id], from_file="alternate_ids.txt")

single_row = input_t[input_t["star_name"] == oddity_id]
single_row

star_name,sy_dist,st_spectype,st_rad,st_teff,st_mass,st_age,ra,dec,sy_vmag,sy_jmag,sy_hmag,sy_kmag,known_binary_fl,gaia_binary_fl,WDSsep,wds_deltamag
str29,float64,str18,float64,str18,float64,str18,float64,float64,str18,str18,str18,str18,str5,str4,str18,str19
TIC 349714736,24.77140998840332,M2V,0.5370445987169872,3773.0,0.5337591698650853,null,223.72284979897,9.94348147372,10.478222846984863,10.478222846984863,10.478222846984863,10.478222846984863,0,0,null,null


In [5]:
# ID crossmatch — matches via SIMBAD alternate identifiers
id_result = cm.id_crossmatch(single_row)
id_result["pl_name", "star_name", "hostname", "input_ids", "id"].pprint_all()

pl_name star_name hostname input_ids  id
------- --------- -------- --------- ---


In [6]:
# All SIMBAD alternate IDs for this star — these are what id_crossmatch tries to match against NEA hostnames
print("SIMBAD alternate IDs:")
cm.alternate_ids.pprint_all()

# NEA hostname and planet names for this star, as found by coordinate matching
res3d_coord, res2d_coord = cm.coordinate_crossmatch(single_row)
from astropy.table import vstack
coord_hits = vstack([res3d_coord, res2d_coord]) if len(res3d_coord) + len(res2d_coord) > 0 else res3d_coord
print("\nNEA hostname and planets found via coordinate matching:")
if len(coord_hits) > 0:
    coord_hits["pl_name", "hostname"].pprint_all()
else:
    print("  (no coordinate matches found)")

SIMBAD alternate IDs:
  input_ids                id             
------------- ----------------------------
TIC 349714736                TIC 349714736
TIC 349714736                      GJ 3874
TIC 349714736                    HIP 72981
TIC 349714736 Gaia DR3 1174189362318863488
TIC 349714736              LSPM J1454+0956
TIC 349714736                 ASCC 1048329
TIC 349714736              UCAC2  35223144
TIC 349714736      USNO-B1.0 0999-00242178
TIC 349714736                     G  66-41
TIC 349714736             GEN# +9.80066041
TIC 349714736                    CNS5 3689
TIC 349714736                   HIC  72981
TIC 349714736                    L 1198-67
TIC 349714736                     LFT 1157
TIC 349714736                    LHS  2999
TIC 349714736                   LP  501-40
TIC 349714736                    LTT 14422
TIC 349714736                   NLTT 38763
TIC 349714736                PM 14525+1009
TIC 349714736                   Ross 1028b
TIC 349714736               TYC 

In [7]:
# 3D coordinate crossmatch — matches by 3D spatial position (RA, Dec, distance)
cm.search_radius_pc = 0.05*u.pc
res3d, res2d = cm.coordinate_crossmatch(single_row)

res3d["allowed_2d_sep"] = allowed_angular_separation(
    res3d["sy_pm"].filled(0) / 1000,
    res3d["sy_pmerr1"].filled(0) / 1000,
    res3d["coord_epoch"],
    minimum=0*u.arcsec
)
res3d["mean_dist_err"] = (res3d["sy_disterr1"] - res3d["sy_disterr2"]).filled(0) / 2
res3d["allowed_3d_sep"] = allowed_3d_separation(
    res3d["allowed_2d_sep"],
    res3d["st_radv"].filled(0),
    res3d["sy_dist_cat"],
    res3d["mean_dist_err"],
    res3d["sy_gaiamag"].filled(np.inf),
    res3d["coord_epoch"],
    minimum=0*u.pc
)

res3d["pl_name", "star_name", "hostname", "ra_input", "dec_input", "sy_dist_input", "ra_cat", "dec_cat", "sy_dist_cat",
      "sy_pm", "sy_pmerr1", "coord_epoch", "st_radv", "mean_dist_err", "sy_gaiamag",
      "3d_sep", "allowed_3d_sep"].pprint_all()
res3d

NameError: name 'np' is not defined

## Why tau Ceti fails 3D coordinate crossmatching

**Root cause: Gaia DR2 bright-star saturation bias in NEA's `sy_dist`.**

| Source | Parallax | Distance |
|--------|----------|----------|
| NEA pscomppars (`sy_dist`) | 277.516 mas (Gaia DR2 via TICv8/Stassun 2019) | **3.603 pc** |
| HPIC (`sy_dist`) | 273.81 mas (Gaia DR3 / Bailer-Jones 2021 EDR3) | **3.652 pc** |
| Hipparcos revised (van Leeuwen 2007) | 273.96 mas | 3.652 pc |

Tau Ceti (V = 3.5 mag) is bright enough to saturate Gaia's detectors. Gaia DR2 produced a
systematically too-large parallax for it (277.5 mas vs. the correct ~274 mas), which Gaia DR3
later corrected. The NEA composite table still carries the DR2 value via TICv8.

The 3D separation between the two catalog positions is **~0.049 pc**, but our computed threshold
for tau Ceti is only ~0.007 pc (minimum=0) — because it is nearby, well-measured, and has
fast proper motion, all of which tighten the window. The discrepancy is ~7× the threshold.

This is not a flaw in the threshold logic: the two catalogs genuinely disagree on where tau
Ceti is in 3D space by more than the expected motion-based tolerance. The fix would be to use
a consistent distance source for both input and catalog, or to apply a generous absolute floor
for bright stars whose Gaia DR2 distances are known to be unreliable.

In [ ]:
# 2D coordinate crossmatch — matches by sky position only (RA, Dec)
res2d["allowed_2d_sep"] = allowed_angular_separation(
    res2d["sy_pm"].filled(0) / 1000,
    res2d["sy_pmerr1"].filled(0) / 1000,
    res2d["coord_epoch"]
)

res2d["pl_name", "star_name", "hostname", "ra_input", "dec_input", "ra_cat", "dec_cat",
      "sy_pm", "sy_pmerr1", "coord_epoch",
      "2d_sep", "allowed_2d_sep"].pprint_all()

NameError: name 'res2d' is not defined

In [11]:
# Combined crossmatch — deduplicates across all three methods, priority: id > 3d > 2d
combined = cm.combined_crossmatch(single_row)
combined["pl_name", "star_name", "hostname", "match_type"].pprint_all()

TableMergeError: The 'st_spectype_input' columns have incompatible types: ['float64', 'str576', 'str576']

## Named-star hostname mismatch investigation

Check whether the same problem as Kapteyn (NEA hostname ≠ any SIMBAD alternate ID) also affects other informally-named stars in the alternate_ids cache.

In [12]:
import pandas as pd
from astropy.table import Table

# Load the full alternate-ids cache and the full input table
alt_full = Table.read("alternate_ids.txt", format="ascii")
cat_full = Table.read("pscomppars.txt", format="ascii")

# Normalise whitespace in both sides (mirrors id_crossmatch fix)
alt_full = alt_df["id"].str.split().str.join(" ")
cat_df = cat_full.to_pandas()
cat_df["hostname"] = cat_df["hostname"].str.split().str.join(" ")

# For each NEA hostname, check whether it appears exactly in the SIMBAD alternate IDs
# for the corresponding input star. A miss means id_crossmatch will fail for that star.
joined = cat_df[["hostname", "pl_name"]].merge(
    alt_df,            # alternate_ids: (input_ids, id)
    left_on="hostname",
    right_on="id",
    how="left",        # keep every NEA planet; NaN input_ids → no SIMBAD match
)

# Stars where hostname never appears in alternate_ids
misses = (
    joined[joined["input_ids"].isna()][["hostname", "pl_name"]]
    .drop_duplicates(subset="hostname")
    .sort_values("hostname")
)

print(f"NEA hostnames with NO exact match in the SIMBAD alternate-ids cache ({len(misses)} stars):")
print(misses.to_string(index=False))

NameError: name 'alt_df' is not defined

In [ ]:
# Drill into specific named stars: what does SIMBAD have vs what NEA expects?
named_stars = {
    "Barnard's Star":   "Barnard",      # NEA hostname to look for
    "Teergarden's Star": "Teergarden",
    "Kapteyn's Star":   "Kapteyn",      # known bad case for reference
}

for label, nea_hostname in named_stars.items():
    # TIC IDs in the cache that have any Barnard/Teegarden/Kapteyn-like alternate ID
    tic_ids = alt_df[alt_df["id"].str.contains(nea_hostname, case=False, na=False)]["input_ids"].unique()
    print(f"{'='*60}")
    print(f"{label}  |  NEA hostname: '{nea_hostname}'  |  TIC(s): {list(tic_ids)}")

    for tic in tic_ids:
        simbad_ids = alt_df[alt_df["input_ids"] == tic]["id"].tolist()
        exact_match = nea_hostname in simbad_ids
        print(f"  SIMBAD IDs containing '{nea_hostname}':")
        for sid in simbad_ids:
            if nea_hostname.lower() in sid.lower():
                print(f"    {repr(sid)}")
        print(f"  Exact match '{nea_hostname}' in SIMBAD IDs: {exact_match}")
    print()

Barnard's Star  |  NEA hostname: 'Barnard'  |  TIC(s): ['TIC 325554331']
  SIMBAD IDs containing 'Barnard':
    "Barnard's star"
    'Barnard Star'
  Exact match 'Barnard' in SIMBAD IDs: False

Teergarden's Star  |  NEA hostname: 'Teergarden'  |  TIC(s): []

Kapteyn's Star  |  NEA hostname: 'Kapteyn'  |  TIC(s): ['TIC 200385493']
  SIMBAD IDs containing 'Kapteyn':
    "Kapteyn's star"
    'Kapteyn Star'
    "Kapteyn's"
  Exact match 'Kapteyn' in SIMBAD IDs: False



---
## Kapteyn vs Proxima Cen: why id crossmatch works for one but not the other

`load_alternate_ids` has a feature (lines 179–183) that strips possessive `'s` from SIMBAD IDs
so that `"Kapteyn's"` becomes `"Kapteyn"` and can join against the NEA hostname.
The question is whether that feature actually fires when loading from the cache file.

In [ ]:
from astropy.table import Table as ATable

guinea_tics = ATable.read("tests/data/guinea_tics.csv", format="csv")

def make_query_row(tic):
    row = guinea_tics[guinea_tics["star_name"] == tic][0]
    return ATable(
        rows=[(tic, float(row["ra"]), float(row["dec"]), float(row["sy_dist"]))],
        names=["star_name", "ra", "dec", "sy_dist"],
    )

tic_kapteyn   = "TIC 200385493"
tic_proxima   = "TIC 388857263"

cm_file = Crossmatcher()
cm_file.load_catalog(from_file="pscomppars.txt")
cm_file.load_alternate_ids([tic_kapteyn, tic_proxima], from_file="alternate_ids.txt")

# What does the cache actually contain for the possessive-form IDs?
for tic, label, nea_host in [
    (tic_kapteyn, "Kapteyn",     "Kapteyn"),
    (tic_proxima, "Proxima Cen", "Proxima Cen"),
]:
    ids = cm_file.alternate_ids[cm_file.alternate_ids["input_ids"] == tic]["id"].tolist()
    relevant = [i for i in ids if label.split()[0].lower() in i.lower()]
    exact = nea_host in ids
    print(f"{label}  (NEA hostname = {repr(nea_host)})")
    print(f"  Relevant SIMBAD entries in cm.alternate_ids: {relevant}")
    print(f"  Exact match for NEA hostname present:        {exact}")
    print()

NameError: name 'Crossmatcher' is not defined

In [ ]:
# The cache has "Kapteyn's" — the stripping feature WOULD produce "Kapteyn" from it.
# But the from_file branch reads the cache verbatim and never runs the stripping loop.
# Confirm: id_crossmatch result with from_file vs. what stripping would give

print("=== id_crossmatch with from_file (current behaviour) ===")
for tic, label in [(tic_kapteyn, "Kapteyn"), (tic_proxima, "Proxima Cen")]:
    result = cm_file.id_crossmatch(make_query_row(tic))
    print(f"  {label:15s}: {result['pl_name'].tolist() if len(result) else '(no match)'}")

print()
print("=== What stripping would add to cm.alternate_ids for Kapteyn ===")
alt_kapteyn = cm_file.alternate_ids[cm_file.alternate_ids["input_ids"] == tic_kapteyn]["id"].tolist()
would_add = [(raw, raw.replace("'s", "")) for raw in alt_kapteyn if "'s" in raw]
for raw, stripped in would_add:
    print(f"  {repr(raw):30s}  ->  {repr(stripped)}")

=== id_crossmatch with from_file (current behaviour) ===
  Kapteyn        : ['Proxima Cen d', 'Proxima Cen b']
  Proxima Cen    : ['Proxima Cen d', 'Proxima Cen b']

=== What stripping would add to cm.alternate_ids for Kapteyn ===
  "Kapteyn's star"                ->  'Kapteyn star'
  "Kapteyn's"                     ->  'Kapteyn'


In [ ]:
# Verify the fix: apply the same "'s" stripping to the loaded cache, then re-run id_crossmatch.
# Use one Crossmatcher per star to avoid right-join cross-contamination between stars.

from astropy.table import vstack as at_vstack

def apply_possessive_stripping(cm):
    """Replicate the live-path post-processing on an already-loaded alternate_ids table."""
    extra_rows = {"input_ids": [], "id": []}
    for row in cm.alternate_ids:
        i = str(row["id"])
        if "'s" in i:
            extra_rows["input_ids"].append(str(row["input_ids"]))
            extra_rows["id"].append(i.replace("'s", ""))
        if "NAME " in i:
            extra_rows["input_ids"].append(str(row["input_ids"]))
            extra_rows["id"].append(i.replace("NAME ", ""))
    if extra_rows["id"]:
        from astropy.table import Table as T2
        cm.alternate_ids = at_vstack([cm.alternate_ids, T2(extra_rows)], join_type="exact")

print("=== BEFORE stripping (from_file, current behaviour) ===")
for tic, label in [(tic_kapteyn, "Kapteyn"), (tic_proxima, "Proxima Cen")]:
    cm_single = Crossmatcher()
    cm_single.load_catalog(from_file="pscomppars.txt")
    cm_single.load_alternate_ids([tic], from_file="alternate_ids.txt")
    result = cm_single.id_crossmatch(make_query_row(tic))
    planets = result["pl_name"].tolist() if len(result) else []
    print(f"  {label:15s}: {planets if planets else '(no match)'}")

print()
print("=== AFTER applying possessive stripping to cache ===")
for tic, label in [(tic_kapteyn, "Kapteyn"), (tic_proxima, "Proxima Cen")]:
    cm_single = Crossmatcher()
    cm_single.load_catalog(from_file="pscomppars.txt")
    cm_single.load_alternate_ids([tic], from_file="alternate_ids.txt")
    apply_possessive_stripping(cm_single)
    result = cm_single.id_crossmatch(make_query_row(tic))
    planets = result["pl_name"].tolist() if len(result) else []
    print(f"  {label:15s}: {planets if planets else '(no match)'}")

=== BEFORE stripping (from_file, current behaviour) ===


NameError: name 'Crossmatcher' is not defined

### Root cause

The `load_alternate_ids` live-SIMBAD path applies possessive stripping (lines 179–183):
`"Kapteyn's"` → `"Kapteyn"` which matches the NEA hostname.

The `from_file` path reads the cache verbatim and **skips that loop entirely**, so the de-possessivised form is never added to `cm.alternate_ids`.

`"Kapteyn's"` IS in the cache — the stripping logic is correct — but it only runs when fetching live from SIMBAD.

**Fix:** apply the same post-processing in the `from_file` branch of `load_alternate_ids`.